In [30]:
!pip install rank-bm25 sentence-transformers transformers accelerate -q faiss-cpu

import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # silencia avisos verbosos da biblioteca

PASTA_DADOS = Path("data")        # a "base de conhecimento" da empresa fictícia
PASTA_INDICE = Path("indice")     # artefatos que o pipeline vai gerar
MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Setup pronto — GPU disponível:", torch.cuda.is_available())

Setup pronto — GPU disponível: True


In [31]:
# ==================================================
# FUNÇÕES DO PIPELINE
# ==================================================

# #RECURSIVE CHUNKING
# def dividir_em_chunks(texto, tamanho=400, overlap=80, separadores=None):
#     if separadores is None:
#         separadores = ["\n\n", "\n", ". ", " "]

#     # Se o texto já couber no tamanho, retorna ele
#     if len(texto) <= tamanho:
#         return [texto.strip()] if texto.strip() else []

#     # Encontra o primeiro separador disponível no texto
#     separador = next((s for s in separadores if s in texto), "")

#     if not separador:
#         # Se não houver separadores, faz o corte bruto por tamanho com overlap
#         chunks, inicio = [], 0
#         while inicio < len(texto):
#             fim = min(inicio + tamanho, len(texto))
#             pedaco = texto[inicio:fim].strip()
#             if pedaco:
#                 chunks.append(pedaco)
#             if fim == len(texto):
#                 break
#             inicio = fim - overlap
#         return chunks

#     # Divide o texto pelo separador escolhido
#     partes = texto.split(separador)
#     chunks = []
#     chunk_atual = ""

#     for parte in partes:
#         candidato = f"{chunk_atual}{separador}{parte}" if chunk_atual else parte

#         if len(candidato) <= tamanho:
#             chunk_atual = candidato
#         else:
#             if chunk_atual:
#                 chunks.append(chunk_atual.strip())

#             # Se a parte isolada for maior que o tamanho, aplica recursão com os próximos separadores
#             if len(parte) > tamanho:
#                 idx = separadores.index(separador) + 1
#                 proximos = separadores[idx:]
#                 chunks.extend(dividir_em_chunks_recursivo(parte, tamanho, overlap, proximos))
#                 chunk_atual = ""
#             else:
#                 chunk_atual = parte

#     if chunk_atual:
#         chunks.append(chunk_atual.strip())

#     return [c for c in chunks if c]

# #FIXED SIZE CHUNKING
# def dividir_em_chunks(texto, tamanho=400, overlap=80):
#     """Chunking de tamanho fixo com overlap e fronteira natural."""
#     chunks, inicio = [], 0
#     while inicio < len(texto):
#         fim = min(inicio + tamanho, len(texto))
#         if fim < len(texto):
#             janela = texto[inicio:fim]
#             corte = max(janela.rfind("\n\n"), janela.rfind(". "))
#             if corte > tamanho // 2:           # só recua se sobrar chunk útil
#                 fim = inicio + corte + 1
#         pedaco = texto[inicio:fim].strip()
#         if pedaco:
#             chunks.append(pedaco)
#         if fim == len(texto):
#             break
#         inicio = fim - overlap                  # o próximo chunk repete o final
#     return chunks

#SEMANTIC CHUNKING
def dividir_em_chunks(texto, modelo_nome='all-MiniLM-L6-v2', limiar_percentil=80):
    """
    Realiza semantic chunking dividindo o texto com base na distância
    semântica entre frases consecutivas.
    """
    # 1. Carrega o modelo de embeddings (leve e rápido)
    modelo = SentenceTransformer(modelo_nome)

    # 2. Divide o texto em frases básicas (exemplo simples usando ponto final)
    # Em produção, bibliotecas como NLTK ou spaCy fazem isso de forma mais robusta.
    frases = [f.strip() + "." for f in texto.split(". ") if f.strip()]

    if not frases:
        return []

    if len(frases) == 1:
        return frases

    # 3. Gera os embeddings para cada frase
    embeddings = modelo.encode(frases)

    # 4. Calcula a distância de cosseno entre frases consecutivas
    def cosseno_distancia(a, b):
        return 1.0 - (np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    distancias = []
    for i in range(len(embeddings) - 1):
        dist = cosseno_distancia(embeddings[i], embeddings[i + 1])
        distancias.append(dist)

    # 5. Define o limiar de corte baseado em um percentil das distâncias
    # Variações maiores que o percentil escolhido marcam a quebra de chunk
    limiar = np.percentile(distancias, limiar_percentil)

    # 6. Agrupa as frases em chunks com base nos pontos de corte
    chunks = []
    chunk_atual = [frases[0]]

    for i, dist in enumerate(distancias):
        if dist > limiar:
            # A mudança de contexto foi alta, fecha o chunk atual e inicia um novo
            chunks.append(" ".join(chunk_atual))
            chunk_atual = [frases[i + 1]]
        else:
            chunk_atual.append(frases[i + 1])

    if chunk_atual:
        chunks.append(" ".join(chunk_atual))

    return chunks


def tokenizar_lexical(texto):
    """Tokenização simples para o BM25: minúsculas + palavras alfanuméricas."""
    # Escolha técnica: BM25 é uma técnica clássica que considera frequência do termo, raridade da palavra e tamanho do documento.
    return re.findall(r"[a-z0-9À-ÿ\-]+", texto.lower())


def rrf(rankings, k=60):
    """Reciprocal Rank Fusion: funde rankings somando 1/(k + posição)."""

    pontos = {}
    for ranking in rankings:
        for posicao, idx in enumerate(ranking):
            pontos[idx] = pontos.get(idx, 0.0) + 1.0 / (k + posicao + 1)
    return sorted(pontos.items(), key=lambda par: -par[1])


print("Funções do pipeline definidas.")



Funções do pipeline definidas.


In [32]:
# ==================================================
# INGESTÃO E CHUNKING
# ==================================================

print("INGESTÃO — lendo as colunas específicas dos arquivos CSV de reviews")
documentos = []

for arq in sorted(PASTA_DADOS.glob("*.csv")):
    # Lê apenas as colunas desejadas usando pandas
        df = pd.read_csv(arq, usecols=["review_score", "review_comment_title", "review_comment_message"])

        # Opcional: Tratar valores nulos preenchendo com vazios para evitar erros ao concatenar
        df["review_comment_title"] = df["review_comment_title"].fillna("")
        df["review_comment_message"] = df["review_comment_message"].fillna("")

        # Cria um texto unificado por linha combinando nota, título e mensagem
        # Exemplo: "Nota: 5. Título: Excelente. Mensagem: Produto chegou muito rápido."
        textos_reviews = []
        for _, row in df.iterrows():
            texto_formatado = f"Nota: {row['review_score']}. Título: {row['review_comment_title']}. Mensagem: {row['review_comment_message']}."
            textos_reviews.append(texto_formatado)

        texto_completo_arquivo = " \n".join(textos_reviews)
        documentos.append((arq.stem, texto_completo_arquivo))

        print(f"  {arq.stem:22s} {len(texto_completo_arquivo):5d} caracteres lidos")


        print("\nCHUNKING")
chunks = []
for nome, texto in documentos:
    #PARA SEMANTIC CHUNKING
    pedacos = dividir_em_chunks(texto)
    # #PARA TAMANHO FIXO E RECURSIVE, RODAR FUNÇÃO ABAIXO:
    # pedacos = dividir_em_chunks(texto, tamanho=400, overlap=80)
    for i, pedaco in enumerate(pedacos):
        chunks.append({"doc": nome, "chunk_id": f"{nome}#{i}", "texto": pedaco})
    print(f"  {nome:22s} -> {len(pedacos):2d} chunks gerados")


print(f"\nTOTAL: {len(chunks)} chunks")


PASTA_INDICE.mkdir(exist_ok=True)
(PASTA_INDICE / "chunks.json").write_text(
    json.dumps(chunks, ensure_ascii=False, indent=1), encoding="utf-8")
print("\nChunks salvos com sucesso em indice/chunks.json!")





INGESTÃO — lendo as colunas específicas dos arquivos CSV de reviews
  olist_order_reviews_dataset 6125912 caracteres lidos

CHUNKING


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  olist_order_reviews_dataset -> 57144 chunks gerados

TOTAL: 57144 chunks

Chunks salvos com sucesso em indice/chunks.json!


In [27]:
# ==================================================
# EMBEDDINGS E INDEXAÇÃO VETORIAL
# ==================================================

emb = SentenceTransformer(MODELO_EMBEDDINGS)

textos_para_vetorizar = [c["texto"] for c in chunks]

print(f"Gerando embeddings para {len(textos_para_vetorizar)} chunks...")
vetores = emb.encode(
    textos_para_vetorizar,
    normalize_embeddings=True,
    batch_size=128,          # aumentando para lotes maiores para aproveitar paralelismo
    show_progress_bar=True
)

print(f"Matriz de embeddings: {vetores.shape}")
np.save(PASTA_INDICE / "embeddings.npy", vetores)

try:
    import faiss
    indice = faiss.IndexFlatIP(vetores.shape[1])
    indice.add(vetores)
    print(f"\nINDEXAÇÃO — FAISS IndexFlatIP com {indice.ntotal} vetores")

    def buscar_vetorial(consulta, k=3):
        vetor = emb.encode([consulta], normalize_embeddings=True)
        distancias, posicoes = indice.search(vetor, k)
        return list(zip(posicoes[0], distancias[0]))
except ImportError:
    print("\nINDEXAÇÃO — FAISS indisponível; busca exata em numpy")

    def buscar_vetorial(consulta, k=3):
        vetor = emb.encode([consulta], normalize_embeddings=True)[0]
        sims = vetores @ vetor
        top = np.argsort(-sims)[:k]
        return [(int(i), float(sims[i])) for i in top]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Gerando embeddings para 57144 chunks...


Batches:   0%|          | 0/447 [00:00<?, ?it/s]

Matriz de embeddings: (57144, 384)

INDEXAÇÃO — FAISS indisponível; busca exata em numpy


In [29]:
# ==================================================
# BUSCAS (RECUPERAÇÃO)
# ==================================================

bm25 = BM25Okapi([tokenizar_lexical(c["texto"]) for c in chunks])
print("Índice BM25 montado (estatísticas de termos, sem rede neural).")

def ranking_bm25(consulta):
    return list(np.argsort(-bm25.get_scores(tokenizar_lexical(consulta))))

def ranking_vetorial(consulta):
    vetor = emb.encode([consulta], normalize_embeddings=True)[0]
    return list(np.argsort(-(vetores @ vetor)))

def mostrar(titulo, ranking, k=3):
    print(f"\n  {titulo}")
    for posicao in ranking[:k]:
        c = chunks[posicao]
        print(f"    [{c['chunk_id']:22s}]  \"{c['texto'][:70]}...\"")

CONSULTAS = [
    ("Consulta buscando experiências positivas",
     "O que os clientes mais elogiam em suas experiências de compra?"),

    ("Consulta buscando reclamações negativas",
     "Quais problemas são relatados por clientes insatisfeitos?"),
]

for descricao, consulta in CONSULTAS:
    print("=" * 70)
    print(f'{descricao}\nCONSULTA: "{consulta}"')
    r_lex, r_vet = ranking_bm25(consulta), ranking_vetorial(consulta)
    mostrar("BM25 (lexical):", r_lex)
    mostrar("Vetorial (semântica):", r_vet)
    fundido = [posicao for posicao, _ in rrf([r_lex[:10], r_vet[:10]])]
    mostrar("HÍBRIDA (RRF — funde os dois rankings):", fundido)


Índice BM25 montado (estatísticas de termos, sem rede neural).
Consulta buscando experiências positivas
CONSULTA: "O que os clientes mais elogiam em suas experiências de compra?"

  BM25 (lexical):
    [olist_order_reviews_dataset#22487]  "Título: Produto errado. Mensagem: Mais responsabilidade com os cliente..."
    [olist_order_reviews_dataset#45138]  "Nota: 5. Título:. Mensagem: A equipe targaryen entrou em contato comig..."
    [olist_order_reviews_dataset#17973]  "Mensagem: compra efetuada,debito autorizado ,pedido cancelado sem expl..."

  Vetorial (semântica):
    [olist_order_reviews_dataset#14072]  "Mensagem: Adorei a experiência de compra...."
    [olist_order_reviews_dataset#39889]  "Mensagem: Super recomendo compro sempre é até o momento mega satisfeit..."
    [olist_order_reviews_dataset#42793]  "Mensagem: Recomendo a compra nessa loja, primam pela pontualidade e qu..."

  HÍBRIDA (RRF — funde os dois rankings):
    [olist_order_reviews_dataset#22487]  "Título: Produto err